# Reasoning Benchmarks with Prompt Paraphrases - Demo\n\nThis notebook demonstrates standardized GSM8K (grade school math) and MBPP (Python coding) reasoning benchmarks augmented with systematic $K=3$ prompt paraphrase variants (synonym substitution, conditional framing, and step-by-step interrogative rephrasings) for robust evaluation of multi-agent LLM systems against prompt variance and semantic perturbation.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# Unconditional installs (packages not pre-installed on Colab)\n_pip('jsonschema==4.26.0')\n\n# Core scientific packages (pre-installed on Colab, install locally to match Colab env)\nif 'google.colab' not in sys.modules:\n    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import os\nimport json\nimport urllib.request\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\n\nprint("Imports completed successfully.")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ca2cc5-resilient-quorum-sensing-multi-agent-rea/main/round-1/dataset-1/demo/mini_demo_data.json"\n\ndef load_data():\n    try:\n        print(f\"Attempting to load data from GitHub URL: {GITHUB_DATA_URL}\")\n        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=10) as response:\n            return json.loads(response.read().decode())\n    except Exception as e:\n        print(f\"GitHub URL load failed ({e}), falling back to local mini_demo_data.json\")\n    \n    if os.path.exists(\"mini_demo_data.json\"):\n        with open(\"mini_demo_data.json\", \"r\") as f:\n            return json.load(f)\n    raise FileNotFoundError(\"Could not load mini_demo_data.json from GitHub or local directory.\")

In [ ]:
data = load_data()\nprint(f\"Loaded dataset structure with {len(data.get('datasets', []))} benchmark categories.\")

## Configuration & Tunable Parameters\n\nDefine configuration parameters for dataset inspection and paraphrase evaluation.

In [ ]:
# Config tunable parameters\nMAX_EXAMPLES_PER_DATASET = 5\nPARAPHRASE_COUNT = 3\nEVAL_SIMULATION_ITERATIONS = 1

## Dataset Exploration & Paraphrase Inspection\n\nWe inspect the loaded benchmark records, including original inputs, reference solutions, and the generated $K=3$ paraphrase variants (synonym replacement, conditional framing, and step-by-step guidance).

In [ ]:
records_summary = []\n\nfor ds_group in data.get(\"datasets\", []):\n    ds_name = ds_group.get(\"dataset\")\n    examples = ds_group.get(\"examples\", [])\n    print(f\"\\n--- Benchmark: {ds_name} ({len(examples)} examples loaded) ---\")\n    \n    for ex in examples[:MAX_EXAMPLES_PER_DATASET]:\n        inp = ex.get(\"input\", \"\")\n        out = ex.get(\"output\", \"\")\n        p1 = ex.get(\"metadata_paraphrase_1\", \"\")\n        p2 = ex.get(\"metadata_paraphrase_2\", \"\")\n        p3 = ex.get(\"metadata_paraphrase_3\", \"\")\n        \n        records_summary.append({\n            \"Dataset\": ds_name,\n            \"Input Length\": len(inp),\n            \"Output Length\": len(out),\n            \"P1 Length\": len(p1),\n            \"P2 Length\": len(p2),\n            \"P3 Length\": len(p3)\n        })\n\ndf_summary = pd.DataFrame(records_summary)\nprint("\nDataset Summary Table:")\nprint(df_summary.to_string(index=False))

## Paraphrase Variance Analysis & Visualization\n\nWe analyze character lengths and linguistic expansion across the original prompts and the three paraphrase variants, visualizing the perturbation distribution.

In [ ]:
plt.figure(figsize=(10, 5))\nx = np.arange(len(df_summary))\nwidth = 0.18\n\nplt.bar(x - 1.5*width, df_summary["Input Length"], width, label="Original Input", color="skyblue")\nplt.bar(x - 0.5*width, df_summary["P1 Length"], width, label="Paraphrase 1 (Synonym)", color="salmon")\nplt.bar(x + 0.5*width, df_summary["P2 Length"], width, label="Paraphrase 2 (Conditional)", color="orange")\nplt.bar(x + 1.5*width, df_summary["P3 Length"], width, label="Paraphrase 3 (Step-by-Step)", color="green")\n\nplt.xlabel("Example Index Across Benchmarks")\nplt.ylabel("Character Length")\nplt.title("Prompt Paraphrase Length Variation Analysis")\nplt.xticks(x, [f"{row['Dataset']}-{i}" for i, row in df_summary.iterrows()])\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint("\nVisualization generated successfully. Demo complete!")